# Step 11: high-value triage (redact, don't filter)

Workspace fevm-workshop-guy, 2026-11-09, conversation 4443667514456316.
Branch step11/2026-11-09_fevm-workshop-guy_conv-4443667514456316, from main dc114b5.

Section A is the before state, captured read-only on 2026-11-10 04:12:45 UTC (2026-11-09 US time) before any change, and kept here as text so a later run cannot overwrite it. Sections B onward are live checks; each prints PASS or FAIL with the measured value.

## A. Before state (read-only, 2026-11-10 04:12:45 UTC)

**A.1 Row counts through UC (fe_bar_lakebase.fraud_dash.*, as databricks_reader_17063)**

| object | rows |
| --- | --- |
| case_summary | 222 |
| case_actions_daily | 9 |
| score_distribution | 31 |
| exposure | 6 |
| baseline_comparison | 2 |
| triage_queue | 24,201 |
| model_thresholds | 1 |

**A.2 Through UC, amount > 10000:** triage_queue 0, case_summary 0 (expected 0).

Supporting probe through UC: fe_bar_lakebase.fraud_gov.txn_features returns 463,881 rows, 0 with amount > 10000; fe_bar_lakebase.fraud.txn_features has 751 rows with amount > 10000. The fraud_gov row rule (pg_has_role(current_user, 'fraud_pii')) is false for the reader, so all 751 are hidden on the UC path.

**A.3 Postgres as fraud_triage_agent (fraud_gov):** current_user = fraud_triage_agent; flagged (block or review) total 24,906; flagged with amount > 10000 = **705**; best risk rank among them = **231**. 24,906 minus 705 = 24,201 = UC triage_queue rows.

| txn_id | amount | action | risk_score | rank | cases |
| --- | --- | --- | --- | --- | --- |
| TXN-0189627 | 14,581.03 | block | 0.9882 | 231 | 0 |
| TXN-0142590 | 10,877.10 | review | 0.9447 | 3,933 | 4 (all closed) |

**A.4 Genie benchmarks that read case_summary or triage_queue (benchmark SQL run through UC)**

| benchmark | result |
| --- | --- |
| Why was TXN-0234209 flagged? | 1 row: block, 0.98441; SHAP is_new_device 1.0 (+2.5724), geo_distance_from_home 450.63 (+1.8741), is_online 1.0 (+1.2001) |
| Which case queue has the most cases with a block recommendation? | auto_block 11 |
| How many cases were created per day and queue in the week starting October 19, 2026? | 10-20 analyst_review 1; 10-21 analyst_review 56, auto_block 7; 10-23 analyst_review 54, auto_block 3 |
| Which 5 customers have the most flagged transactions in the triage queue? | CUST-01399 20, CUST-00167 17, CUST-03895 16, CUST-04867 16, CUST-06862 16 |
| How many open cases are in each queue? | auto_block 10 |

**A.5 Dashboard dataset "Case KPIs" (ds_kpi_cases):** cases_created_today 0, open_cases 10, analyst_review_backlog 0, visible_cases 222, as_of_date_utc 2026-11-10.

**A.6 exposure and baseline_comparison (reference for Part E)**

| strategy | flagged | fraud caught | fraud $ caught | false positives | fraud $ missed |
| --- | --- | --- | --- | --- | --- |
| model | 24,201 | 6,571 | 7,010,482.03 | 17,630 | 975,725.89 |
| baseline_rule | 33,723 | 2,767 | 933,889.13 | 30,956 | 7,052,318.79 |

exposure (action, is_fraud, txn_count, total_amount): approve/false 435,924 59,930,426.63; approve/true 3,756 975,725.89; block/false 736 589,241.33; block/true 1,278 524,286.80; review/false 16,894 19,798,228.68; review/true 5,293 6,486,195.23.

**A.7 App ACL (GET /api/2.0/permissions/apps/fe-bar-fraud-triage)**

| principal | level | inherited |
| --- | --- | --- |
| alex.owen@databricks.com | CAN_MANAGE | no |
| group pii | CAN_USE | no |
| group admins | CAN_MANAGE | yes, from /apps |

Note: group pii CAN_USE was already present when this was captured. The expected before state (alex.owen and admins only) did not match the live ACL.

In [0]:
# Same pinned packages the App installs, so psycopg and the App SQL run here unchanged.
%pip install -r /Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo/app/requirements.txt --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import sys, json, time, re, ast, datetime
from pathlib import Path

REPO = Path("/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")
sys.path.insert(0, str(REPO))
from lakebase_cases import LakebaseCases
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
APP, APP_SP = "fe-bar-fraud-triage", "8f239f44-078a-4f8f-90f1-4c1b0bcdc26c"
SPACE, DASH, WH = "01f1d7b464d510b9aa6cde48300787ca", "01f1ce7f835a1e81b05ebb664bf0994c", "9bbc027402111984"
DEPLOYMENT_ID = "01f1dcc1df121891906c3a2c6df85a1a"   # Part C redeploy, 2026-11-10 04:47 UTC
CONV_ID = "4443667514456316"

RESULTS = []
def check(name, ok, value):
    RESULTS.append((name, bool(ok), value))
    print(f"{'PASS' if ok else 'FAIL'}  {name}: {value}")

def uc(q):
    return [r.asDict() for r in spark.sql(q).collect()]

RUN_TS = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
print("run started (UTC):", RUN_TS)

run started (UTC): 2026-11-10T05:20:49+00:00


In [0]:
agent = LakebaseCases(pg_role="fraud_triage_agent")
pg_hv = agent.query("""
    SELECT count(*) AS n FROM fraud_gov.txn_risk_current r
    JOIN fraud_gov.txn_features tf ON tf.txn_id = r.txn_id
    WHERE r.recommended_action IN ('block', 'review') AND tf.amount > 10000""")[0]["n"]

t0 = time.time()
tq = uc("""SELECT count(*) AS n, count_if(is_high_value) AS hv,
                 count_if(is_high_value AND amount IS NOT NULL) AS hv_amt,
                 count_if(is_high_value AND amount_band <> 'over_10k') AS bad_band
          FROM fe_bar_lakebase.fraud_dash.triage_queue""")[0]
t_tq = time.time() - t0
t0 = time.time()
cs = uc("""SELECT count(*) AS n, count_if(is_high_value) AS hv,
                 count_if(is_high_value AND amount IS NOT NULL) AS hv_amt
          FROM fe_bar_lakebase.fraud_dash.case_summary""")[0]
t_cs = time.time() - t0

check("E.1 UC triage_queue is_high_value rows = Postgres flagged amount > 10000 (expect 705)",
      tq["hv"] == pg_hv, f"UC {tq['hv']} vs Postgres (fraud_triage_agent) {pg_hv}; triage_queue rows {tq['n']}")
check("E.2 UC non-null amount where is_high_value (triage_queue + case_summary)",
      tq["hv_amt"] == 0 and cs["hv_amt"] == 0 and tq["bad_band"] == 0,
      f"triage_queue {tq['hv_amt']}, case_summary {cs['hv_amt']} (case_summary rows {cs['n']}, high-value {cs['hv']})")

leak = uc("""
    SELECT count(*) AS leaks, count(*) FILTER (WHERE s.shap IS NULL) AS shap_missing
    FROM fe_bar_lakebase.fraud_dash.triage_queue
    LATERAL VIEW explode(from_json(top_shap_signals, 'array<struct<feature:string,value:double,shap:double>>')) e AS s
    WHERE is_high_value AND s.feature IN ('amount', 'amt_to_cust_avg_30d', 'amt_to_monthly_spend')
      AND s.value IS NOT NULL""")[0]
feat = uc("""
    SELECT count(*) AS n, count_if(s.shap IS NOT NULL) AS with_shap
    FROM fe_bar_lakebase.fraud_dash.triage_queue
    LATERAL VIEW explode(from_json(top_shap_signals, 'array<struct<feature:string,value:double,shap:double>>')) e AS s
    WHERE is_high_value AND s.feature IN ('amount', 'amt_to_cust_avg_30d', 'amt_to_monthly_spend')""")[0]
check("E.3 UC non-null SHAP value for amount / amt_to_cust_avg_30d / amt_to_monthly_spend on high-value rows",
      leak["leaks"] == 0 and feat["n"] > 0 and feat["with_shap"] == feat["n"],
      f"{leak['leaks']} non-null values; {feat['n']} amount-derived signals on high-value rows, all keep feature name + shap: {feat['with_shap'] == feat['n']}")

r = uc("""SELECT txn_id, amount, amount_band, is_high_value, risk_rank, recommended_action, case_status
          FROM fe_bar_lakebase.fraud_dash.triage_queue WHERE txn_id = 'TXN-0189627'""")
check("E.4 UC TXN-0189627 in triage_queue with amount NULL and amount_band over_10k",
      len(r) == 1 and r[0]["amount"] is None and r[0]["amount_band"] == "over_10k", r)
check("E.4b both changed views return in under 10 s through UC",
      t_tq < 10 and t_cs < 10, f"triage_queue {t_tq:.1f} s, case_summary {t_cs:.1f} s")

PASS  E.1 UC triage_queue is_high_value rows = Postgres flagged amount > 10000 (expect 705): UC 705 vs Postgres (fraud_triage_agent) 705; triage_queue rows 24906
PASS  E.2 UC non-null amount where is_high_value (triage_queue + case_summary): triage_queue 0, case_summary 0 (case_summary rows 229, high-value 7)
PASS  E.3 UC non-null SHAP value for amount / amt_to_cust_avg_30d / amt_to_monthly_spend on high-value rows: 0 non-null values; 1688 amount-derived signals on high-value rows, all keep feature name + shap: True
PASS  E.4 UC TXN-0189627 in triage_queue with amount NULL and amount_band over_10k: [{'txn_id': 'TXN-0189627', 'amount': None, 'amount_band': 'over_10k', 'is_high_value': True, 'risk_rank': 231, 'recommended_action': 'block', 'case_status': 'open'}]
PASS  E.4b both changed views return in under 10 s through UC: triage_queue 2.8 s, case_summary 3.2 s


In [0]:
EXPECTED_BASELINE = {
    "model":         dict(rows_evaluated=463881, flagged_count=24201, fraud_caught_count=6571, fraud_amount_caught=7010482.03,
                          false_positive_count=17630, fraud_missed_count=3756, fraud_amount_missed=975725.89, total_fraud_amount=7986207.92),
    "baseline_rule": dict(rows_evaluated=463881, flagged_count=33723, fraud_caught_count=2767, fraud_amount_caught=933889.13,
                          false_positive_count=30956, fraud_missed_count=7560, fraud_amount_missed=7052318.79, total_fraud_amount=7986207.92),
}
EXPECTED_EXPOSURE = {  # (action, is_fraud): (txn_count, total_amount, fraud_amount_caught, fraud_amount_missed)
    ("approve", False): (435924, 59930426.63, 0.0, 0.0),
    ("approve", True):  (3756, 975725.89, 0.0, 975725.89),
    ("block", False):   (736, 589241.33, 0.0, 0.0),
    ("block", True):    (1278, 524286.8, 524286.8, 0.0),
    ("review", False):  (16894, 19798228.68, 0.0, 0.0),
    ("review", True):   (5293, 6486195.23, 6486195.23, 0.0),
}
bl = {r["strategy"]: r for r in uc("SELECT * FROM fe_bar_lakebase.fraud_dash.baseline_comparison")}
bl_ok = all(all(bl[s][k] == v for k, v in exp.items()) for s, exp in EXPECTED_BASELINE.items()) and len(bl) == 2
ex = {(r["recommended_action"], r["is_fraud"]): (r["txn_count"], r["total_amount"], r["fraud_amount_caught"], r["fraud_amount_missed"])
      for r in uc("SELECT * FROM fe_bar_lakebase.fraud_dash.exposure")}
check("E.5a baseline_comparison identical to Part A", bl_ok,
      f"model ${bl['model']['fraud_amount_caught']:,.2f} vs rule ${bl['baseline_rule']['fraud_amount_caught']:,.2f} caught")
check("E.5b exposure identical to Part A", ex == EXPECTED_EXPOSURE, f"{len(ex)} cells, total ${sum(v[1] for v in ex.values()):,.2f}")

PASS  E.5a baseline_comparison identical to Part A: model $7,010,482.03 vs rule $933,889.13 caught
PASS  E.5b exposure identical to Part A: 6 cells, total $88,304,104.56


In [0]:
admin   = LakebaseCases()                      # my login, for ground truth
analyst = LakebaseCases(pg_role="fraud_analyst")

hv_cases = {r["case_id"] for r in admin.query("""
    SELECT c.case_id::text AS case_id FROM cases.fraud_cases c
    JOIN fraud.txn_features tf ON tf.txn_id = c.txn_id WHERE tf.amount > 10000""")}
an_cases = {r["case_id"] for r in analyst.query("SELECT case_id::text AS case_id FROM cases.fraud_cases")}
check("E.6 fraud_analyst: high-value cases still hidden by RLS", len(hv_cases) > 0 and not (hv_cases & an_cases),
      f"analyst sees {len(an_cases)} cases, {len(hv_cases & an_cases)} of {len(hv_cases)} high-value cases")

# Part A snapshot (2026-11-10 04:20 UTC): 19 grant rows, 13 distinct (schema, table, privilege).
READER_GRANTS_BEFORE = {
    ("cases", "case_actions", "SELECT"), ("cases", "fraud_cases", "SELECT"),
    ("fraud_dash", "baseline_comparison", "SELECT"), ("fraud_dash", "case_actions_daily", "SELECT"),
    ("fraud_dash", "case_summary", "SELECT"), ("fraud_dash", "exposure", "SELECT"),
    ("fraud_dash", "model_thresholds", "SELECT"), ("fraud_dash", "score_distribution", "SELECT"),
    ("fraud_dash", "triage_queue", "SELECT"), ("fraud_gov", "customer_risk_profile", "SELECT"),
    ("fraud_gov", "risk_score_calibration", "SELECT"), ("fraud_gov", "txn_features", "SELECT"),
    ("fraud_gov", "txn_risk_current", "SELECT"),
}
rows = admin.query("""SELECT table_schema, table_name, privilege_type FROM information_schema.role_table_grants
                      WHERE grantee = 'databricks_reader_17063'""")
now = {(r["table_schema"], r["table_name"], r["privilege_type"]) for r in rows}
check("E.7 reader role_table_grants identical to before (diff)", now == READER_GRANTS_BEFORE and len(rows) == 19,
      f"rows {len(rows)}, added {sorted(now - READER_GRANTS_BEFORE)}, removed {sorted(READER_GRANTS_BEFORE - now)}")
mem = admin.query("""SELECT r.rolname FROM pg_auth_members a JOIN pg_roles r ON r.oid = a.roleid
                     JOIN pg_roles m ON m.oid = a.member WHERE m.rolname = 'databricks_reader_17063'""")
check("E.7b reader role memberships (expect none)", len(mem) == 0, mem)
views = admin.query("""SELECT c.relname, pg_get_userbyid(c.relowner) AS owner, c.reloptions FROM pg_class c
                       WHERE c.relnamespace = 'fraud_dash'::regnamespace AND c.relkind = 'v' ORDER BY 1""")
check("E.7c all six fraud_dash views owned by fraud_lead with security_barrier",
      len(views) == 6 and all(v["owner"] == "fraud_lead" and "security_barrier=true" in (v["reloptions"] or []) for v in views),
      [v["relname"] for v in views])
lead = admin.query("""SELECT has_table_privilege('fraud_lead', 'fraud.txn_features', 'SELECT') AS f,
                             has_table_privilege('fraud_lead', 'fraud.txn_risk_current', 'SELECT') AS r""")[0]
check("E.7d approved Step 11 grant present: fraud_lead SELECT on fraud.txn_features, fraud.txn_risk_current",
      lead["f"] and lead["r"], lead)

PASS  E.6 fraud_analyst: high-value cases still hidden by RLS: analyst sees 211 cases, 0 of 7 high-value cases
PASS  E.7 reader role_table_grants identical to before (diff): rows 19, added [], removed []
PASS  E.7b reader role memberships (expect none): []
PASS  E.7c all six fraud_dash views owned by fraud_lead with security_barrier: ['baseline_comparison', 'case_actions_daily', 'case_summary', 'exposure', 'score_distribution', 'triage_queue']
PASS  E.7d approved Step 11 grant present: fraud_lead SELECT on fraud.txn_features, fraud.txn_risk_current: {'f': True, 'r': True}


In [0]:
app = w.api_client.do("GET", f"/api/2.0/apps/{APP}")
check("E.8a App RUNNING on the Step 11 deployment",
      app["app_status"]["state"] == "RUNNING" and app["active_deployment"]["deployment_id"] == DEPLOYMENT_ID,
      f"{app['app_status']['state']}, active deployment {app['active_deployment']['deployment_id']} "
      f"({app['active_deployment']['status']['state']}, {app['active_deployment']['create_time']})")

acl = w.api_client.do("GET", f"/api/2.0/permissions/apps/{APP}")   # permissions API takes the App name
entries = []
for e in acl["access_control_list"]:
    who = e.get("user_name") or e.get("group_name") or e.get("service_principal_name")
    for p in e["all_permissions"]:
        entries.append((who, p["permission_level"], p.get("inherited", False)))
for x in entries:
    print("  ", x)
check("E.8b App ACL: group pii CAN_USE, no 'users' grant",
      ("pii", "CAN_USE", False) in entries and not any(e[0] == "users" for e in entries), entries)

# Filled in from the non_pii test user's attempt (None = not tried).
NON_PII_APP_DENIED = None
print("non_pii test user App URL attempt:", {None: "not tried", True: "denied", False: "NOT denied"}[NON_PII_APP_DENIED])

PASS  E.8a App RUNNING on the Step 11 deployment: RUNNING, active deployment 01f1dcc1df121891906c3a2c6df85a1a (SUCCEEDED, 2026-11-10T04:47:45Z)
   ('alex.owen@databricks.com', 'CAN_MANAGE', False)
   ('pii', 'CAN_USE', False)
   ('admins', 'CAN_MANAGE', True)
PASS  E.8b App ACL: group pii CAN_USE, no 'users' grant: [('alex.owen@databricks.com', 'CAN_MANAGE', False), ('pii', 'CAN_USE', False), ('admins', 'CAN_MANAGE', True)]
non_pii test user App URL attempt: not tried


In [0]:
# The user clicks in the App; this cell checks the database. TXN-0189627 had 0 cases before the test (Part A).
# The App's own SQL strings are loaded from app/app.py, so these checks run the exact deployed queries.
# User's report of what the App showed (2026-11-10, Transaction Detail txn_id box = TXN-0189627).
# The first attempt used the picker default TXN-0147293 and returned its existing case 82b12d25 (no insert).
USER_OBSERVED = {
    "explain_returned": True,
    "explain_summary": "user-observed: Explain returned an explanation for TXN-0189627",
    "open_case_id": "d2a1d4a1-7f84-4ed9-b9aa-4146c9f57592",
    "second_click_same_case": True,
    "txn_0142590_rank": 3933,
    "hv_only_rows": 50,
}

src = (REPO / "app" / "app.py").read_text()
ns = {}
for node in ast.parse(src).body:
    if isinstance(node, ast.Assign) and getattr(node.targets[0], "id", "") in ("_QUEUE_SQL", "_LOOKUP_SQL"):
        exec(compile(ast.Module([node], []), "app.py", "exec"), ns)

lk = agent.query(ns["_LOOKUP_SQL"], ("TXN-0189627",))
check("E.9a App search TXN-0189627 finds it (exact amount, PII tier)", len(lk) == 1,
      [(x["txn_id"], x["amount"], x["recommended_action"], x["risk_rank"], x["case_status"]) for x in lk])

lead_lc = LakebaseCases(pg_role="fraud_lead")
cases = lead_lc.query("""SELECT case_id::text AS case_id, status, assigned_queue, recommended_action,
                                db_inserted_by, created_at FROM cases.fraud_cases WHERE txn_id = 'TXN-0189627'
                         ORDER BY created_at""")
for c in cases:
    print("  ", c)
check("E.9b Open case wrote one case for TXN-0189627 with db_inserted_by = App role",
      len(cases) == 1 and cases[0]["db_inserted_by"] == APP_SP
      and cases[0]["case_id"] == USER_OBSERVED["open_case_id"],
      f"{len(cases)} case(s); case_id {cases[0]['case_id'] if cases else None} "
      f"(App showed {USER_OBSERVED['open_case_id']}); db_inserted_by {cases[0]['db_inserted_by'] if cases else None}")
check("E.9c second Open case click added no duplicate", len(cases) == 1 and USER_OBSERVED.get("second_click_same_case") is True,
      f"{len(cases)} case(s); App showed the same case ID on the second click: {USER_OBSERVED.get('second_click_same_case')}")
check("E.9d Explain returned an explanation (user-observed; App logs need interactive OAuth)",
      USER_OBSERVED.get("explain_returned") is True, USER_OBSERVED.get("explain_summary"))

lk2 = agent.query(ns["_LOOKUP_SQL"], ("TXN-0142590",))
check("E.9e App search TXN-0142590 finds it at rank 3,933",
      len(lk2) == 1 and lk2[0]["risk_rank"] == 3933 and USER_OBSERVED.get("txn_0142590_rank") in (None, 3933),
      f"query rank {lk2[0]['risk_rank'] if lk2 else None}; App showed {USER_OBSERVED.get('txn_0142590_rank')}")

hv = agent.query(ns["_QUEUE_SQL"], (["block", "review"], True, ["auto_block", "analyst_review"], "all", "all", 50))
check("E.9f High-value only returns 50 rows, all amount > 10000",
      len(hv) == 50 and all(x["amount"] > 10000 for x in hv) and USER_OBSERVED.get("hv_only_rows") in (None, 50),
      f"query {len(hv)} rows, min amount {min(x['amount'] for x in hv):,.2f}, first {hv[0]['txn_id']} rank {hv[0]['risk_rank']}; "
      f"App showed {USER_OBSERVED.get('hv_only_rows')} rows")
CASE_ID_TXN_0189627 = cases[0]["case_id"] if cases else None
print("case ID for TXN-0189627:", CASE_ID_TXN_0189627)

PASS  E.9a App search TXN-0189627 finds it (exact amount, PII tier): [('TXN-0189627', 14581.03, 'block', 231, 'open')]
   {'case_id': 'd2a1d4a1-7f84-4ed9-b9aa-4146c9f57592', 'status': 'open', 'assigned_queue': 'auto_block', 'recommended_action': 'block', 'db_inserted_by': '8f239f44-078a-4f8f-90f1-4c1b0bcdc26c', 'created_at': datetime.datetime(2026, 11, 10, 5, 11, 19, 310138, tzinfo=zoneinfo.ZoneInfo(key='GMT'))}
PASS  E.9b Open case wrote one case for TXN-0189627 with db_inserted_by = App role: 1 case(s); case_id d2a1d4a1-7f84-4ed9-b9aa-4146c9f57592 (App showed d2a1d4a1-7f84-4ed9-b9aa-4146c9f57592); db_inserted_by 8f239f44-078a-4f8f-90f1-4c1b0bcdc26c
PASS  E.9c second Open case click added no duplicate: 1 case(s); App showed the same case ID on the second click: True
PASS  E.9d Explain returned an explanation (user-observed; App logs need interactive OAuth): user-observed: Explain returned an explanation for TXN-0189627
PASS  E.9e App search TXN-0142590 finds it at rank 3,933: query ra

In [0]:
import requests

# Re-running B.7-B.9 alone must not double-count: drop earlier E.10-E.12 results before re-checking.
RESULTS[:] = [r for r in RESULTS if not r[0].startswith(("E.10", "E.11", "E.12"))]

space = json.loads(w.api_client.do("GET", f"/api/2.0/genie/spaces/{SPACE}",
                                   query={"include_serialized_space": "true"})["serialized_space"])
BENCH = sorted(space["benchmarks"]["questions"], key=lambda q: q["question"][0])
base = f"{w.config.host.rstrip('/')}/api/2.0/genie/spaces/{SPACE}"
hdr = lambda: {**w.config.authenticate(), "Content-Type": "application/json"}

def ask(q):
    r = requests.post(f"{base}/start-conversation", headers=hdr(), json={"content": q}, timeout=60)
    r.raise_for_status()
    d = r.json()
    cid, mid = d["conversation_id"], d["message_id"]
    m = {}
    for _ in range(150):
        m = requests.get(f"{base}/conversations/{cid}/messages/{mid}", headers=hdr(), timeout=60).json()
        if m.get("status") in ("COMPLETED", "FAILED", "CANCELLED", "QUERY_RESULT_EXPIRED"):
            break
        time.sleep(2)
    sql, att, text = None, None, ""
    for a in m.get("attachments") or []:
        if a.get("query"):
            sql, att = a["query"].get("query"), a.get("attachment_id")
        if a.get("text"):
            text = a["text"].get("content", "")
    rows = []
    if att:
        qr = requests.get(f"{base}/conversations/{cid}/messages/{mid}/attachments/{att}/query-result",
                          headers=hdr(), timeout=60).json()
        rows = ((qr.get("statement_response") or {}).get("result") or {}).get("data_array") or []
    return dict(conv=cid, msg=mid, status=m.get("status"), sql=sql, text=text, rows=rows)

def tokens(values):
    out = set()
    for v in values:
        if v is None:
            continue
        s = str(v)
        if s[:1] in "[{":
            try:
                out |= tokens([x for d in json.loads(s) for x in d.values()])
                continue
            except Exception:
                pass
        if re.fullmatch(r"\d{4}-\d{2}-\d{2}.*", s):
            out.add(s[:10]); continue
        try:
            out.add(f"{round(float(s), 2):.2f}")
        except ValueError:
            out.add(s.strip().lower())
    return out

GENIE = []
for b in BENCH:
    q = b["question"][0]
    exp_sql = " ".join(b["answer"][0].get("content") or [])
    t0 = time.time()
    g = ask(q)
    secs = round(time.time() - t0)
    if exp_sql:
        exp = tokens([v for r in uc(exp_sql) for v in r.values()])
        got = tokens([v for r in g["rows"] for v in r])
        ok = g["status"] == "COMPLETED" and exp <= got
        note = "all expected values present" if ok else f"missing {sorted(exp - got)[:6]}"
    else:   # refusal benchmark: must return no result rows
        ok = g["status"] == "COMPLETED" and not g["rows"]
        note = "refused, no rows" if ok else f"{len(g['rows'])} rows returned"
    src_tables = sorted(set(re.findall(r"fraud_dash\.\w+|risk_score_calibration", g["sql"] or "")))
    GENIE.append(dict(question=q, ok=ok, secs=secs, conv=g["conv"], msg=g["msg"], sources=src_tables, note=note))
    print(f"{'PASS' if ok else 'FAIL'} {secs:>3}s  {q}  | {src_tables}  | {note}  | conv {g['conv']} msg {g['msg']}")

n_pass = sum(x["ok"] for x in GENIE)
new_q = {"How many high-value transactions are in the triage queue?",
         "Show the top 5 high-value block transactions in the triage queue with their amount band"}
check("E.10a Genie as pii: both new high-value benchmarks pass",
      all(x["ok"] for x in GENIE if x["question"] in new_q) and sum(x["question"] in new_q for x in GENIE) == 2,
      [(x["question"], x["ok"]) for x in GENIE if x["question"] in new_q])
check("E.10b Genie as pii: all benchmarks", n_pass == len(GENIE), f"{n_pass}/{len(GENIE)} PASS")
# Deferred by the user on 2026-11-10: the non_pii test user's Genie run and App URL attempt are not part of
# this run. Recorded as DEFERRED (not PASS) so the gap stays visible; run them later and turn this into a check.
DEFERRED = [
    "E.10c Genie as non_pii test user: both new high-value benchmarks pass (deferred by the user)",
    "E.8c non_pii test user denied at the App URL (deferred by the user)",
]
for d in DEFERRED:
    print("DEFERRED", d)

PASS  16s  How many cases were created per day and queue in the week starting October 19, 2026?  | []  | all expected values present  | conv 01f1dcc9bebb1707810ffcfc64dcb978 msg 01f1dcc9becf14dbb976514d4e5edc97
PASS  14s  How many cases were opened between October 20 and October 23, 2026?  | []  | all expected values present  | conv 01f1dcc9ca1d101e9931ebbfe7d4bc78 msg 01f1dcc9ca271f6b948189dcbfe4ba2c
PASS  14s  How many fraud dollars did the model miss, and how many false positives did it generate?  | []  | all expected values present  | conv 01f1dcc9d49811d2b5dc19a34787f931 msg 01f1dcc9d4a2116ebcca5608be9c42ed
PASS  14s  How many high-value transactions are in the triage queue?  | []  | all expected values present  | conv 01f1dcc9dfd31e8594debe0498f2f0a8 msg 01f1dcc9dfdc102ea529e41b60004926
PASS  18s  How many open cases are in each queue?  | []  | all expected values present  | conv 01f1dcc9ea06197a808680263ab95ae7 msg 01f1dcc9ea0f197abfadf6ce7eeafb80
PASS  18s  How much fraud did t

In [0]:
# Export calls (live objects, not hand-edited):
#   GET /api/2.0/genie/spaces/{SPACE}?include_serialized_space=true
#   GET /api/2.0/lakeview/dashboards/{DASH}  (+ /published to confirm the draft is the published revision)
live = w.api_client.do("GET", f"/api/2.0/genie/spaces/{SPACE}", query={"include_serialized_space": "true"})
genie_out = {
    "space_id": live["space_id"], "title": live["title"], "description": live["description"],
    "warehouse_id": live["warehouse_id"],
    "exported_at": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "exported_in_conversation": CONV_ID,
    "serialized_space": json.loads(live["serialized_space"]),
}
gp = REPO / "genie" / "fraud_risk_ops_genie.json"
gp.write_text(json.dumps(genie_out, indent=2, ensure_ascii=False) + "\n")

dash = w.api_client.do("GET", f"/api/2.0/lakeview/dashboards/{DASH}")
pub = w.api_client.do("GET", f"/api/2.0/lakeview/dashboards/{DASH}/published")
dp = REPO / "dashboards" / "fe_bar_fraud_ops.lvdash.json"
dp.write_text(json.dumps(json.loads(dash["serialized_dashboard"]), indent=2, ensure_ascii=False))

print("draft update_time", dash["update_time"], "| published revision", pub["revision_create_time"],
      "| embed_credentials", pub["embed_credentials"])
for p in (gp, dp):
    t = p.read_text()
    hits = [pat for pat in ("dapi", "eyJ", "ghp_", "password=", "Bearer") if pat in t]
    print(f"wrote {p.relative_to(REPO)} ({len(t):,} chars); token hits {hits}; em-dash {t.count(chr(0x2014))}")
check("E.11 dashboard published with embedded credentials after the last draft update",
      pub["embed_credentials"] and pub["revision_create_time"] >= dash["update_time"],
      f"{len(json.loads(dash['serialized_dashboard'])['datasets'])} datasets")
check("E.12 Genie export has 13 benchmarks and the redaction note",
      len(genie_out["serialized_space"]["benchmarks"]["questions"]) == 13 and "amount redacted" in genie_out["description"],
      len(genie_out["serialized_space"]["benchmarks"]["questions"]))

draft update_time 2026-11-10T04:54:07.342Z | published revision 2026-11-10T04:54:07.495Z | embed_credentials True
wrote genie/fraud_risk_ops_genie.json (30,132 chars); token hits []; em-dash 0
wrote dashboards/fe_bar_fraud_ops.lvdash.json (56,831 chars); token hits []; em-dash 0
PASS  E.11 dashboard published with embedded credentials after the last draft update: 11 datasets
PASS  E.12 Genie export has 13 benchmarks and the redaction note: 13


In [0]:
rec = LakebaseCases().query("""SELECT case_id::text AS case_id, txn_id, status, assigned_queue, created_by,
                                     db_inserted_by, created_at FROM cases.fraud_cases WHERE case_id = %s::uuid""",
                            ("d2a1d4a1-7f84-4ed9-b9aa-4146c9f57592",))
print("Live-test case record:", rec, "\n")

n_pass = sum(ok for _, ok, _ in RESULTS)
print(f"Step 11 validation, run {RUN_TS}: {n_pass} PASS / {len(RESULTS) - n_pass} FAIL of {len(RESULTS)} checks\n")
for name, ok, _ in RESULTS:
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print(f"\nNot run ({len(DEFERRED)}, deferred by the user, not counted as PASS):")
for d in DEFERRED:
    print(f"DEFERRED  {d}")

Live-test case record: [{'case_id': 'd2a1d4a1-7f84-4ed9-b9aa-4146c9f57592', 'txn_id': 'TXN-0189627', 'status': 'open', 'assigned_queue': 'auto_block', 'created_by': 'fraud_triage_copilot:agent', 'db_inserted_by': '8f239f44-078a-4f8f-90f1-4c1b0bcdc26c', 'created_at': datetime.datetime(2026, 11, 10, 5, 11, 19, 310138, tzinfo=zoneinfo.ZoneInfo(key='GMT'))}] 

Step 11 validation, run 2026-11-10T05:20:49+00:00: 24 PASS / 0 FAIL of 24 checks

PASS  E.1 UC triage_queue is_high_value rows = Postgres flagged amount > 10000 (expect 705)
PASS  E.2 UC non-null amount where is_high_value (triage_queue + case_summary)
PASS  E.3 UC non-null SHAP value for amount / amt_to_cust_avg_30d / amt_to_monthly_spend on high-value rows
PASS  E.4 UC TXN-0189627 in triage_queue with amount NULL and amount_band over_10k
PASS  E.4b both changed views return in under 10 s through UC
PASS  E.5a baseline_comparison identical to Part A
PASS  E.5b exposure identical to Part A
PASS  E.6 fraud_analyst: high-value cases st